In [ ]:
# Input for Version 2: family destination and route planning
user_query = """
Plan a family vacation and determine:

- Which places are appropriate for the family to visit.
- In what sequence the destinations should be covered.
- Which transportation options are available between destinations.
- Whether airline, train, bus, or taxi is the most suitable option for each journey.
- Whether the itinerary meets comfort, time, and budget constraints.
"""

print(user_query)

### Hierarchical Planning Example: Family Vacation to Italy

Scenario Overview: Planning an international vacation involves many interconnected decisions such as budgeting, travel arrangements, accommodation, local transport, and activity selection. Without structured planning, the process becomes overwhelming and time-consuming.

Approach: This use case demonstrates Hierarchical Planning, where a high-level goal is broken down into organized sub-tasks and executable steps. The AI agent analyzes the final objective—“Plan a family vacation to Italy”—and transforms it into a multi-layer plan:

🔹 Level 1: High-Level Goal

→ Plan a complete 7-day family trip to Italy.

🔹 Level 2: Sub-Goals

Set travel budget

Book flights

Reserve accommodations

Plan local travel

Create sightseeing/activity schedule

Organize food + dietary considerations

🔹 Level 3: Actionable Steps

Example breakdown:

| Sub-goal        | Action Steps                                                                                           |
| --------------- | ------------------------------------------------------------------------------------------------------ |
| Budgeting       | Estimate flight cost, hotel pricing, activities, food allowance                                        |
| Flights         | Compare airlines, choose departure dates, book tickets                                                 |
| Accommodation   | Evaluate Airbnb vs hotels, check proximity to landmarks, confirm reservation                           |
| Activities      | Create daily itinerary (Rome → Florence → Venice), book museum passes, identify child-friendly options |
| Local Transport | Lookup rail passes, taxi availability, or rental car rules                                             |
| Food Planning   | Identify restaurants, check dietary needs, make reservations                                           |



Outcome

The result is a structured travel plan that transforms a vague intention into clear, prioritized execution steps.

✔ Reduced planning stress
✔ Efficient cost and time management
✔ Smart task sequencing
✔ Clear blueprint for execution

In [1]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6


In [7]:
# Warning control
import warnings
warnings.filterwarnings('ignore')
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


2026-11-01 19:30:17,416 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 32
2026-11-01 19:30:17,430 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 33
2026-11-01 19:30:17,431 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 34


In [8]:
# CrewAI: Hierarchical Planning — Family Vacation to Italy
# Agents: Planner, Researcher, Budgeter, ItineraryMaker, Coordinator
# Dynamic inputs: vacation_goal, dates, family_size, preferences

from crewai import Agent, Task, Crew

# ----------------------
# Agents
# ----------------------

planner = Agent(
    role="High-Level Trip Planner",
    goal="Decompose the vacation goal into a hierarchy of sub-goals and prioritized tasks.",
    backstory="You break down large travel goals into clear, ordered subtasks and define success criteria.",
    llm=llm,
    verbose=True
)

researcher = Agent(
    role="Travel Researcher",
    goal="Find practical options for flights, accommodations, and local transport given constraints.",
    backstory="You are skilled at quickly comparing travel options and returning concise recommendations.",
    llm=llm,
    verbose=True
)

budgeter = Agent(
    role="Travel Budget Analyst",
    goal="Estimate costs across flights, hotels, transport, food, and activities and recommend a budget.",
    backstory="You create realistic cost estimates and suggest cost-saving tradeoffs for families.",
    llm=llm,
    verbose=True
)

itinerary_maker = Agent(
    role="Itinerary Creator",
    goal="Produce a detailed day-by-day itinerary aligned with family preferences, pace, and child-friendly options.",
    backstory="You craft balanced itineraries with travel time, rest, and high-impact experiences.",
    llm=llm,
    verbose=True
)

coordinator = Agent(
    role="Trip Coordinator",
    goal="Assemble outputs from other agents into a single cohesive travel plan and checklist.",
    backstory="You synthesize plans into an actionable final deliverable and list next steps for booking.",
    llm=llm,
    verbose=True
)


In [9]:
# Tasks (use dynamic placeholders)

decompose_task = Task(
    description=(
        "Decompose the high-level goal:\n"
        "Goal: {vacation_goal}\n"
        "Dates: {dates}\n"
        "Family size: {family_size}\n"
        "Preferences: {preferences}\n\n"
        "Produce a hierarchical plan with levels:\n"
        "Level 1: Major pillars (travel, lodging, transport, activities, budget)\n"
        "Level 2: For each pillar, list 3-5 concrete sub-tasks\n"
        "Level 3: For the top-priority subtasks, suggest success criteria and timelines.\n\n"
        "Return as a structured list or JSON-like text."
    ),
    expected_output="Hierarchical plan (pillars -> subtasks -> success criteria)",
    agent=planner
)

research_task = Task(
    description=(
        "Research travel options given:\n"
        "Goal: {vacation_goal}\n"
        "Dates: {dates}\n"
        "Family size: {family_size}\n"
        "Preferences: {preferences}\n\n"
        "Return concise recommendations for:\n"
        "- 2 flight options (price tier and rationale)\n"
        "- 2 accommodation types (hotel vs apartment) and one sample neighborhood each\n"
        "- Local transport options (rail passes, car rental notes)\n\n"
        "If any option requires online bookings or external checks, flag it with 'ACTION REQUIRED'."
    ),
    expected_output="Research summary with 2-3 recommended options per category",
    agent=researcher
)

budget_task = Task(
    description=(
        "Create a budget estimate for the trip described:\n"
        "Goal: {vacation_goal}\n"
        "Dates: {dates}\n"
        "Family size: {family_size}\n"
        "Preferences: {preferences}\n\n"
        "Include estimated costs (per category): Flights, Accommodation (per night), Local transport, Food (per day), Activities, Contingency.\n"
        "Give a 'Total estimated budget' and two cost-optimization suggestions."
    ),
    expected_output="Budget table and cost-saving recommendations",
    agent=budgeter
)

itinerary_task = Task(
    description=(
        "Build a 7-day sample itinerary (adjust to {dates} length if needed) for the family:\n"
        "Goal: {vacation_goal}\n"
        "Family size: {family_size}\n"
        "Preferences: {preferences}\n\n"
        "For each day list:\n"
        "- Morning / Afternoon / Evening items\n"
        "- Estimated travel time between stops\n"
        "- Child-friendly notes and booking needs\n"
        "- Time buffers for rest\n\n"
        "Return a balanced, realistic daily schedule."
    ),
    expected_output="Day-by-day itinerary with notes",
    agent=itinerary_maker
)

coordinate_task = Task(
    description=(
        "Assemble final travel plan using previous outputs.\n"
        "Inputs: hierarchical plan, research summary, budget estimate, itinerary.\n\n"
        "Produce a final deliverable containing:\n"
        "- Executive summary (1-2 paragraphs)\n"
        "- Actionable booking checklist (flights, hotels, passes, reservations)\n"
        "- Timeline for when to book each item\n"
        "- Risk notes and contingencies\n"
        "- Final total estimated cost\n\n"
        "Format clearly for sharing with family."
    ),
    expected_output="Final trip plan & checklist",
    agent=coordinator
)

In [10]:
# Build Crew

vacation_crew = Crew(
    llm=llm,
    agents=[planner, researcher, budgeter, itinerary_maker, coordinator],
    tasks=[decompose_task, research_task, budget_task, itinerary_task, coordinate_task],
    verbose=2
)

2026-11-01 19:43:30,788 - 8606633536 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [11]:
vacation_goal = "Plan a 7-day family vacation to Italy focusing on Rome, Florence, and Venice."
dates = "2027-06-10 to 2027-06-17"
family_size = "2 adults, 2 children (ages 8 and 5)"
preferences = "Moderate budget, kid-friendly activities, avoid long overnight travel, prefer central accommodations"


In [13]:
inputs = {
        "vacation_goal": vacation_goal,
        "dates": dates,
        "family_size": family_size,
        "preferences": preferences
    }

result = vacation_crew.kickoff(inputs)
print("\n--- FINAL VACATION PLAN ---\n")
print(result)

 [DEBUG]: == Working Agent: High-Level Trip Planner
 [INFO]: == Starting Task: Decompose the high-level goal:
Goal: Plan a 7-day family vacation to Italy focusing on Rome, Florence, and Venice.
Dates: 2027-06-10 to 2027-06-17
Family size: 2 adults, 2 children (ages 8 and 5)
Preferences: Moderate budget, kid-friendly activities, avoid long overnight travel, prefer central accommodations

Produce a hierarchical plan with levels:
Level 1: Major pillars (travel, lodging, transport, activities, budget)
Level 2: For each pillar, list 3-5 concrete sub-tasks
Level 3: For the top-priority subtasks, suggest success criteria and timelines.

Return as a structured list or JSON-like text.


> Entering new CrewAgentExecutor chain...
Thought: I need to break down the vacation goal into a structured hierarchical plan that includes major pillars, subtasks, and success criteria. I will start by identifying the major pillars of the vacation plan.

Action: Delegate work to co-worker
Action Input: {
    "c

In [14]:
print("\n--- FINAL VACATION PLAN ---\n")
print(result)


--- FINAL VACATION PLAN ---

---

# Family Trip to Italy: June 10 - June 17, 2027

## Executive Summary
This travel plan outlines a 7-day family trip to Italy, covering the vibrant cities of Rome, Florence, and Venice. The itinerary is designed to balance cultural experiences with engaging activities for children, ensuring a memorable vacation. Each day includes structured activities, rest periods, and family-friendly dining options. The estimated total cost for the trip is $6,290, which encompasses flights, accommodations, meals, transportation, activities, and miscellaneous expenses. 

## Actionable Booking Checklist
1. **Flights**
   - Book round-trip flights from New York City (JFK) to Rome (FCO) and return from Venice (VCE) to New York City (JFK).
   - Estimated Cost: $3,200 for 4 people.

2. **Accommodations**
   - **Rome (June 10 - June 13)**: Book Hotel Artemide.
   - **Florence (June 13 - June 15)**: Book Hotel Brunelleschi.
   - **Venice (June 15 - June 17)**: Book Hotel Ant

# Version 2

In [15]:
import os
import json
from getpass import getpass

from crewai import Agent, Task, Crew, Process


# Warning control
import warnings
warnings.filterwarnings('ignore')
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# llm = LLM(
#     model="openai/gpt-4o-mini",
#     temperature=0
# )


2026-11-01 19:59:41,722 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 32
2026-11-01 19:59:41,725 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 33
2026-11-01 19:59:41,726 - 8606633536 - main.py-main:27 - WARNING: Python-dotenv could not parse statement starting at line 34


In [4]:
#from crewai import Agent, Task, Crew, Process

In [16]:

planner_agent = Agent(
    role="Hierarchical Travel Planner",

    goal="""
    Understand user travel requirements and generate
    a structured hierarchical plan.

    Identify destinations, journey legs, constraints,
    subgoals, dependencies and required specialist agents.
    """,

    backstory="""
    You are an experienced travel planning architect.

    You know how to break complex travel requests into
    smaller, manageable tasks.

    You understand family comfort, transportation
    limitations, travel budgets and route planning.

    Your responsibility is PLANNING, not booking or
    executing travel arrangements.
    """,

    llm=llm,
    verbose=True,
    allow_delegation=False
)



planning_task = Task(
    description="""
    Analyze the following travel request:

    {user_query}

    Create a HIERARCHICAL TRAVEL PLAN.

    STEP 1: Understand the primary goal.
    Identify destinations, travelers, dates,
    budget and constraints.

    STEP 2: Identify candidate travel routes.
    Determine which locations need connecting.
    Do not assume the user's destination order
    must be the final route.

    STEP 3: Break the goal into subgoals.
    Examples:
    - Destination suitability research
    - Transportation research
    - Budget estimation
    - Itinerary preparation

    STEP 4: Identify journey legs.
    For each leg, determine which transportation
    agents should investigate possible options.

    Available specialist agents:
    - AirlineAgent
    - TrainAgent
    - BusAgent
    - TaxiAgent
    - DestinationAgent
    - BudgetAgent
    - ItineraryAgent

    STEP 5: Identify dependencies.
    Determine which tasks must finish before
    another task can begin.

    IMPORTANT RULES:
    - Do not execute any tasks.
    - Do not claim live availability or prices.
    - Consider traveler ages and comfort.
    - Respect road-travel limits.
    - Identify missing information.
    - Each task must have a unique ID.
    - Only assign agents from the available list.

    Return ONLY valid JSON in this format:

{{
  "primary_goal": "...",
  "destinations": [],
  "constraints": [],
  "missing_information": [],
  "candidate_routes": [],
  "tasks": [
    {{
      "id": 1,
      "subgoal": "...",
      "agent": "DestinationAgent",
      "description": "...",
      "depends_on": [],
      "success_criteria": "..."
    }}
  ]
}}

    """,

    expected_output="""
    A valid JSON object containing the primary goal,
    destinations, constraints, missing information,
    candidate routes and hierarchical tasks with
    agent assignments, dependencies and success criteria.
    """,

    agent=planner_agent
)


In [17]:

queries = {
    "rome": """
    Plan a 3-day family vacation to Rome
    for two adults and two children aged 5 and 8.
    Include child-friendly attractions.
    """,

    "italy": """
    Plan a 7-day family vacation covering
    Rome, Florence and Venice.

    Family: 2 adults and 2 children aged 5 and 8.
    Budget: Rs. 4 lakh.
    Compare train, bus and taxi options.
    Avoid taxi journeys longer than 4 hours.
    """,

    "senior": """
    Plan a 6-day trip to Rome and Florence
    for two adults and a 72-year-old senior
    with limited mobility.

    Avoid long walks and taxi journeys
    longer than 2 hours.
    """
}





In [4]:
# Keep the custom input defined before Version 2.
# To use a built-in example instead, uncomment one of the following:
# user_query = queries["rome"]
# user_query = queries["italy"]
# user_query = queries["senior"]

In [21]:
user_query = queries["senior"]

In [22]:
planning_crew = Crew(
    agents=[planner_agent],
    tasks=[planning_task],
    process=Process.sequential,
    verbose=True
)

result = planning_crew.kickoff(
    inputs={"user_query": user_query}
)

raw_output = str(getattr(result, "raw", result))
print(raw_output)

2026-11-01 20:04:28,062 - 8606633536 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


 [DEBUG]: == Working Agent: Hierarchical Travel Planner
 [INFO]: == Starting Task: 
    Analyze the following travel request:

    
    Plan a 6-day trip to Rome and Florence
    for two adults and a 72-year-old senior
    with limited mobility.

    Avoid long walks and taxi journeys
    longer than 2 hours.
    

    Create a HIERARCHICAL TRAVEL PLAN.

    STEP 1: Understand the primary goal.
    Identify destinations, travelers, dates,
    budget and constraints.

    STEP 2: Identify candidate travel routes.
    Determine which locations need connecting.
    Do not assume the user's destination order
    must be the final route.

    STEP 3: Break the goal into subgoals.
    Examples:
    - Destination suitability research
    - Transportation research
    - Budget estimation
    - Itinerary preparation

    STEP 4: Identify journey legs.
    For each leg, determine which transportation
    agents should investigate possible options.

    Available specialist agents:
    - AirlineA

In [23]:

raw_output = raw_output.strip()

# Handle an optional markdown code fence
if raw_output.startswith("```"):
    raw_output = raw_output.split("\n", 1)[1]
    raw_output = raw_output.rsplit("```", 1)[0]

plan = json.loads(raw_output.strip())

print("PRIMARY GOAL:")
print(plan["primary_goal"])

print("\nPLANNED TASKS:")

for task in plan["tasks"]:
    print(f'\nTask {task["id"]}')
    print("Agent:", task["agent"])
    print("Subgoal:", task["subgoal"])
    print("Depends on:", task["depends_on"])


PRIMARY GOAL:
Plan a 6-day trip to Rome and Florence for two adults and a 72-year-old senior with limited mobility.

PLANNED TASKS:

Task 1
Agent: DestinationAgent
Subgoal: Destination suitability research
Depends on: []

Task 2
Agent: TrainAgent
Subgoal: Transportation research
Depends on: []

Task 3
Agent: BudgetAgent
Subgoal: Budget estimation
Depends on: []

Task 4
Agent: ItineraryAgent
Subgoal: Itinerary preparation
Depends on: [1, 2, 3]
